In [ ]:
!pip install transformers datasets torch librosa scikit-learn

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import torch

import umap
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import StratifiedKFold, cross_val_score
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler
from datasets import load_dataset, Audio, DatasetDict
from transformers import AutoModelForAudioClassification, AutoFeatureExtractor

In [ ]:
SEED = 42
EMOTIONS = ["angry", "disgust", "fear", "happy", "neutral", "pleasant surprise", "sad"]
COLORS = {e: plt.cm.tab10(i) for i, e in enumerate(EMOTIONS)}   # ίδιο χρώμα ανά συναίσθημα σε όλα τα σχήματα

### Φόρτωση μοντέλων και δεδομένων

In [ ]:
device = "cuda" if torch.cuda.is_available() else "cpu"
model_or = AutoModelForAudioClassification.from_pretrained("facebook/data2vec-audio-base-960h").to(device).eval()
model_ft = AutoModelForAudioClassification.from_pretrained("VasilisAsim/data2vec-finetuned-TESS_full").to(device).eval()

feature_extractor = AutoFeatureExtractor.from_pretrained("VasilisAsim/data2vec-finetuned-TESS_full")

model_or.eval()
model_ft.eval()

models = {
    "Base (data2vec-audio-base-960h)": model_or,
    "Fine-tuned (TESS)": model_ft,
}

In [ ]:
def load_tess_speaker_independent():
    """TESS με speaker-independent διαχωρισμό: OAF = train, YAF = test."""
    dataset = load_dataset("AbstractTTS/TESS", split="train")
    dataset = dataset.cast_column("audio", Audio(decode=False))

    def get_speaker(example):
        filename = example["audio"]["path"]
        if "OAF" in filename:
            return "OAF"
        elif "YAF" in filename:
            return "YAF"
        return "unknown"

    train_dataset = dataset.filter(lambda x: get_speaker(x) == "OAF")
    test_dataset = dataset.filter(lambda x: get_speaker(x) == "YAF")
    train_dataset = train_dataset.cast_column("audio", Audio(sampling_rate=16_000))
    test_dataset = test_dataset.cast_column("audio", Audio(sampling_rate=16_000))
    return DatasetDict({"train": train_dataset, "test": test_dataset})

test_dataset = load_tess_speaker_independent()["test"]

### Εξαγωγή embeddings

In [ ]:
def extract_embeddings(model, dataset):
    """
    Mean pooling της εξόδου του τελευταίου encoder layer (768 διαστάσεις), ένα δείγμα ανά
    forward pass (χωρίς padding). Είναι η αναπαράσταση που δέχεται η κεφαλή ταξινόμησης
    πριν το projector.
    """
    embs, labels = [], []
    for sample in dataset:
        audio = sample["audio"]["array"]
        inputs = feature_extractor(audio, sampling_rate=sample["audio"]["sampling_rate"], return_tensors="pt")
        with torch.no_grad():
            hidden = model.data2vec_audio(inputs["input_values"].to(device)).last_hidden_state
        embs.append(hidden.mean(dim=1).squeeze().cpu().numpy())
        labels.append(sample["emotion"])
    return np.array(embs), np.array(labels)


embeddings = {}
for name, model in models.items():
    embeddings[name], labels = extract_embeddings(model, test_dataset)
print({n: e.shape for n, e in embeddings.items()})

### Συνάρτηση σχεδίασης (αριστερά base, δεξιά fine-tuned)

In [ ]:
def plot_pair(coords, title, xlabel, ylabel):
    """Δύο panels (base / fine-tuned) με κοινά χρώματα συναισθημάτων και κοινή λεζάντα."""
    fig, axes = plt.subplots(1, 2, figsize=(14, 6))
    for ax, (name, xy) in zip(axes, coords.items()):
        for e in EMOTIONS:
            idx = labels == e
            ax.scatter(xy[idx, 0], xy[idx, 1], s=12, alpha=0.8, color=COLORS[e], label=e)
        ax.set_title(name)
        ax.set_xlabel(xlabel)
        ax.set_ylabel(ylabel)
    handles, names = axes[0].get_legend_handles_labels()
    fig.legend(handles, names, loc="lower center", ncol=len(EMOTIONS), markerscale=2)
    fig.suptitle(title)
    fig.tight_layout(rect=(0, 0.06, 1, 0.95))
    plt.show()

### UMAP


In [ ]:
### UMAP

umap_coords = {name: umap.UMAP(n_components=2, n_neighbors=15, min_dist=0.1, metric="cosine",
                               random_state=SEED).fit_transform(emb)
               for name, emb in embeddings.items()}
plot_pair(umap_coords, "UMAP των embeddings (δείγματα YAF)", "UMAP 1", "UMAP 2")